In [ ]:
# download the weaviate client
%pip install -U weaviate-client

In [ ]:
import weaviate, os
from weaviate.config import AdditionalConfig, Timeout
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(override=True)

# Retrieve environment variables
CLUSTER_URL = os.getenv("CLUSTER_URL")
API_KEY = os.getenv("API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Connect to Weaviate
client = weaviate.connect_to_weaviate_cloud(
	cluster_url=CLUSTER_URL,
	auth_credentials=weaviate.auth.AuthApiKey(API_KEY),
	headers={
		"X-OpenAI-Api-Key": OPENAI_API_KEY,
		"X-Cohere-Api-Key": COHERE_API_KEY,
        "X-Goog-Api-Key": GOOGLE_API_KEY
	},
	additional_config=AdditionalConfig(
		timeout=Timeout(init=30, query=60, insert=120)
	)
)

ready = client.is_ready()
server_version = client.get_meta()["version"]
client_version = weaviate.__version__
live = client.is_live()
connected = client.is_connected()

print(f"Weaviate Ready: {ready}")
print(f"Weaviate Client Version: {client_version}")
print(f"Weaviate Server Version: {server_version}")
print(f"Weaviate Live: {client.is_live()}")
print(f"Client Connected: {connected}")

In [ ]:
# Aggregate Objects in a collection
import weaviate.classes.aggregate
# Get the collection
collection = client.collections.use("<COLLECTION_NAME>")
# Perform the aggregation
response = collection.aggregate.over_all()

print(f"Total objects in '{collection.name}' collection: {response.total_count}")

In [ ]:
# List the collection names in Weaviate cluster
try:
    collections = client.collections.list_all()
    if collections:
        print("Collections in Weaviate:")
        # Loop through each collection in the instance
        for collection_name in collections.keys():
            print(f"- {collection_name}")
    else:
        print("No collections found.")
except Exception as e:
    print(f"Error retrieving collections: {e}")

In [ ]:
# Aggregation with groupBy
from weaviate.classes.aggregate import GroupByAggregate

# Get the collection
collection = client.collections.use("<COLLECTION_NAME>")

# Perform the aggregation with groupBy
response = collection.aggregate.over_all(
    group_by=GroupByAggregate(prop="<PROP_NAME>"),
    total_count=True
)

# Print the results
for group in response.groups:
    print(f"Value: {group.grouped_by.value}")
    print(f"Path: {group.grouped_by.prop}")
    print(f"Count: {group.total_count}")

In [ ]:
# Aggregates collection and logs HTTP debug information.
import logging
import datetime
from weaviate.classes.aggregate import GroupByAggregate

# Set up logging to capture HTTP requests
import http.client as http_client
http_client.HTTPConnection.debuglevel = 1

# Configure logging
logging.basicConfig()
logging.getLogger().setLevel(logging.DEBUG)
requests_log = logging.getLogger("requests.packages.urllib3")
requests_log.setLevel(logging.DEBUG)
requests_log.propagate = True

# Record timestamp
current_timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()
print(f"Starting query at: {current_timestamp}")

try:
    # Get the collection
    collection = client.collections.use("<COLLECTION_NAME>")
    
    # Perform the aggregation with groupBy
    response = collection.aggregate.over_all(
        group_by=GroupByAggregate(prop="<PROPERTY_NAME>"),
        total_count=True
    )
    
    # Print the results
    for group in response.groups:
        print(f"Value: {group.grouped_by.value}")
        print(f"Path: {group.grouped_by.prop}")
        print(f"Count: {group.total_count}")
        
except Exception as e:
    print(f"Error occurred: {str(e)}")
    print(f"Error type: {type(e).__name__}")

print("\nInformation:")
print(f"1. Timestamp: {current_timestamp}")
print("2. URL: Check the debug logs above")
print("3. Headers: Check the debug logs above")

In [ ]:
# Aggregate all collections and tenants in the Weaviate instance, and log the results in a DataFrame.

def aggregate_collections(client):
    try:
        collections = client.collections.list_all()

        if not collections:
            return _empty_result()

        collection_count = len(collections)
        total_tenants_count = 0
        empty_collections = 0
        empty_tenants = 0
        total_objects_regular = 0
        total_objects_multitenancy = 0
        empty_collections_list = []
        empty_tenants_details = []
        rows = []

        for collection_name in collections:
            try:
                collection = client.collections.use(collection_name)

                is_multi_tenant = False
                tenants = {}
                try:
                    config = collection.config.get()
                    is_multi_tenant = config.multi_tenancy_config.enabled
                    if is_multi_tenant:
                        tenants = collection.tenants.get() or {}
                except Exception:
                    pass

                if is_multi_tenant:
                    rows.append({
                        "type": "collection",
                        "collection": collection_name,
                        "count": None,
                        "tenant": None,
                        "tenant_count": None,
                    })

                    if not len(tenants):
                        rows.append({
                            "type": "tenant_notice",
                            "collection": None,
                            "count": None,
                            "tenant": "(no tenants exist)",
                            "tenant_count": None,
                        })
                    else:
                        total_tenants_count += len(tenants)
                        collection_tenant_total = 0

                        for tenant_name, tenant in tenants.items():
                            try:
                                tenant_collection = collection.with_tenant(tenant_name)
                                objects_count = tenant_collection.aggregate.over_all(total_count=True).total_count
                                collection_tenant_total += objects_count

                                if objects_count == 0:
                                    empty_tenants += 1
                                    empty_tenants_details.append({
                                        "collection": collection_name,
                                        "tenant": tenant_name,
                                        "count": 0,
                                    })

                                rows.append({
                                    "type": "tenant",
                                    "collection": None,
                                    "count": None,
                                    "tenant": tenant_name,
                                    "tenant_count": objects_count,
                                })
                            except Exception as e:
                                rows.append({
                                    "type": "tenant",
                                    "collection": None,
                                    "count": None,
                                    "tenant": tenant_name,
                                    "tenant_count": f"ERROR: {e}",
                                })

                        total_objects_multitenancy += collection_tenant_total

                else:
                    try:
                        objects_count = collection.aggregate.over_all(total_count=True).total_count
                    except Exception as e:
                        objects_count = f"ERROR: {e}"

                    if isinstance(objects_count, int):
                        if objects_count == 0:
                            empty_collections += 1
                            empty_collections_list.append({
                                "collection": collection_name,
                                "count": 0,
                            })
                        total_objects_regular += objects_count

                    rows.append({
                        "type": "collection",
                        "collection": collection_name,
                        "count": objects_count,
                        "tenant": None,
                        "tenant_count": None,
                    })

            except Exception as e:
                rows.append({
                    "type": "collection",
                    "collection": collection_name,
                    "count": f"ERROR: {e}",
                    "tenant": None,
                    "tenant_count": None,
                })

        return {
            "collection_count": collection_count,
            "total_tenants_count": total_tenants_count,
            "empty_collections": empty_collections,
            "empty_tenants": empty_tenants,
            "total_objects_regular": total_objects_regular,
            "total_objects_multitenancy": total_objects_multitenancy,
            "total_objects_combined": total_objects_regular + total_objects_multitenancy,
            "rows": rows,
            "empty_collections_list": empty_collections_list,
            "empty_tenants_details": empty_tenants_details,
        }

    except Exception as e:
        return {"error": str(e)}


def _empty_result():
    return {
        "collection_count": 0,
        "total_tenants_count": 0,
        "empty_collections": 0,
        "empty_tenants": 0,
        "total_objects_regular": 0,
        "total_objects_multitenancy": 0,
        "total_objects_combined": 0,
        "rows": [],
        "empty_collections_list": [],
        "empty_tenants_details": [],
    }


def print_aggregation_result(result):
    if "error" in result:
        print("Error:", result["error"])
        return

    print("Aggregated Collections Table:")
    print("-" * 45)

    for row in result["rows"]:
        if row["type"] == "collection":
            count = row["count"] if row["count"] is not None else ""
            print(f"  {row['collection']}: {count}")
        elif row["type"] == "tenant":
            print(f"      ↳ {row['tenant']}: {row['tenant_count']}")
        elif row["type"] == "tenant_notice":
            print(f"      ↳ (no tenants exist)")

    print("-" * 45)
    print(f"Total objects:      {result['total_objects_combined']:,}")
    print(f"Collections:        {result['collection_count']}")
    print(f"Empty collections:  {result['empty_collections']}")
    print(f"Total tenants:      {result['total_tenants_count']}")
    print(f"Empty tenants:      {result['empty_tenants']}")

    empty_collection_names = [item["collection"] for item in result.get("empty_collections_list", [])]
    empty_tenant_names = [f"{item['collection']} / {item['tenant']}" for item in result.get("empty_tenants_details", [])]

    print(f"\nEmpty Collections are {empty_collection_names}")
    print(f"Empty Tenants are {empty_tenant_names}")


# ── Run ───────────────────────────────────────────────────────
result = aggregate_collections(client)
print_aggregation_result(result)